# TorqueSense — training on a Colab T4 (spec Section 27)

Two models, both exported to ONNX and recorded in `models/manifest.json` with their SHA-256 (I12):

1. **YOLO11n** — hands and tools, on the 960×540 downscale; shared by every profile.
2. **`threaded_fastener` classifier** — MobileNetV3-Small on 96×96 crops of each inspection point,
   three states: `absent`, `seated`, `fastened`.

Inputs come from `synth/generate.py` (zipped to Drive). Colab is used for parallelism and heat,
not capability: the laptop renders and builds while this trains. Runtime → T4 GPU.

In [ ]:
!pip -q install ultralytics onnx onnxscript
from google.colab import drive; drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/torquesense'
SEED = 20260721
!cp {ROOT}/synth_w14_s20260721.zip /content/ && unzip -q -o /content/synth_w14_s20260721.zip -d /content/data

## 1 · Hands and tools — YOLO11n

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(
    data='/content/data/dataset.yaml',
    epochs=100, imgsz=640, batch=32,        # T4 has 16 GB; 32 fits
    close_mosaic=10, patience=25,
    cache='disk',                            # decode JPEGs once, not every epoch
    project=f'{ROOT}/runs', name='w14',
    save_period=5,                           # survive a Colab disconnect
    seed=SEED, deterministic=True,
)
# after a disconnect: YOLO(f'{ROOT}/runs/w14/weights/last.pt').train(resume=True)
best = YOLO(f'{ROOT}/runs/w14/weights/best.pt')
best.export(format='onnx', opset=17, imgsz=640, simplify=True, dynamic=False)

## 2 · The `threaded_fastener` state classifier — MobileNetV3-Small, 96×96 crops

In [ ]:
import torch, torchvision as tv, torch.nn as nn
from torch.utils.data import DataLoader
torch.manual_seed(SEED)
STATES = ['absent', 'seated', 'fastened']                  # the family's order: rank = index
tf_train = tv.transforms.Compose([tv.transforms.Resize((96, 96)), tv.transforms.ColorJitter(0.3, 0.3, 0.3),
                                  tv.transforms.RandomRotation(15), tv.transforms.ToTensor(),
                                  tv.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
tf_val = tv.transforms.Compose([tv.transforms.Resize((96, 96)), tv.transforms.ToTensor(),
                                tv.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
train = tv.datasets.ImageFolder('/content/data/crops/threaded_fastener/train', tf_train)
val = tv.datasets.ImageFolder('/content/data/crops/threaded_fastener/val', tf_val)
assert train.classes == sorted(STATES), train.classes
net = tv.models.mobilenet_v3_small(weights='DEFAULT')
net.classifier[3] = nn.Linear(net.classifier[3].in_features, len(STATES))
net = net.cuda()
opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, 30)
for epoch in range(30):
    net.train()
    for x, y in DataLoader(train, batch_size=128, shuffle=True, num_workers=2):
        opt.zero_grad(); nn.functional.cross_entropy(net(x.cuda()), y.cuda()).backward(); opt.step()
    sched.step()
    net.eval(); correct = n = 0
    with torch.no_grad():
        for x, y in DataLoader(val, batch_size=256):
            correct += (net(x.cuda()).argmax(1).cpu() == y).sum().item(); n += len(y)
    print(epoch, f'val acc {correct / n:.4f}')
# ImageFolder sorts class names; export with the outputs reordered into the family's order
perm = [train.classes.index(s) for s in STATES]
class Ordered(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return torch.softmax(self.m(x), 1)[:, perm]
torch.onnx.export(Ordered(net).eval().cpu(), torch.zeros(1, 3, 96, 96), f'{ROOT}/threaded_fastener.onnx',
                  opset_version=17, input_names=['crops'], output_names=['probs'],
                  dynamic_axes={'crops': {0: 'n'}, 'probs': {0: 'n'}})   # every crop in view, one batch

## 3 · The model manifest (I12) — copy both files and this JSON to `models/`

In [ ]:
import hashlib, json, pathlib
def sha(p): return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
manifest = {
  'yolo11n_wheel.onnx': {'sha256': sha(f'{ROOT}/runs/w14/weights/best.onnx'), 'version': f'yolo11n-w14-s{SEED}',
                         'trained_from': f'data/synth/<glTF sha>-g3-s{SEED}', 'accuracy_report': 'docs/accuracy.json'},
  'threaded_fastener.onnx': {'sha256': sha(f'{ROOT}/threaded_fastener.onnx'), 'version': f'mnv3s-tf-s{SEED}',
                             'states': STATES, 'input': [96, 96]}}
print(json.dumps(manifest, indent=2))